# Notebook 01 — Data Warehouse: Schema & Load Documentation

This notebook documents the structure of the fraud analytics data warehouse and 
verifies that data loaded correctly. It is the structural reference for the five 
analytical notebooks that follow.

**What this covers:**
- The star-schema design (1 fact table + 6 dimensions)
- Row counts and load verification
- Data profiling and quality checks
- Key design decisions (timestamp reconstruction, "unknown" handling, role-playing dimensions)

**Source:** IEEE-CIS Fraud Detection — 590,540 transactions, Dec 2017 – Jun 2018.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

# Move to project root if running from notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load credentials from .env
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    v = conn.execute(text("SELECT version()")).scalar()
print("Connected to:", v.split(',')[0])

Connected to: PostgreSQL 17.7 on x86_64-windows


## 1. Star Schema Structure

The warehouse uses a classic star schema: a central fact table holding one row per 
transaction, surrounded by six dimension tables that describe the *who, what, when, 
and where* of each transaction.

In [2]:
query_tables = """
SELECT 
    t.table_name,
    (SELECT COUNT(*) FROM information_schema.columns c
     WHERE c.table_schema = 'fraud_dw' AND c.table_name = t.table_name) AS columns,
    CASE t.table_name
        WHEN 'fact_transactions' THEN 'FACT'
        ELSE 'DIMENSION'
    END AS table_type
FROM information_schema.tables t
WHERE t.table_schema = 'fraud_dw'
ORDER BY table_type DESC, t.table_name;
"""
df_tables = pd.read_sql(query_tables, engine)
df_tables

,table_name,columns,table_type
0,fact_transactions,13,FACT
1,dim_card,9,DIMENSION
2,dim_date,12,DIMENSION
3,dim_device,6,DIMENSION
4,dim_email_domain,3,DIMENSION
5,dim_geography,5,DIMENSION
6,dim_product,3,DIMENSION
7,v_scored_transactions,10,DIMENSION


In [3]:
query_counts = """
SELECT 'fact_transactions' AS table_name, COUNT(*) AS row_count FROM fraud_dw.fact_transactions
UNION ALL SELECT 'dim_date',         COUNT(*) FROM fraud_dw.dim_date
UNION ALL SELECT 'dim_card',         COUNT(*) FROM fraud_dw.dim_card
UNION ALL SELECT 'dim_product',      COUNT(*) FROM fraud_dw.dim_product
UNION ALL SELECT 'dim_device',       COUNT(*) FROM fraud_dw.dim_device
UNION ALL SELECT 'dim_geography',    COUNT(*) FROM fraud_dw.dim_geography
UNION ALL SELECT 'dim_email_domain', COUNT(*) FROM fraud_dw.dim_email_domain
ORDER BY row_count DESC;
"""
df_counts = pd.read_sql(query_counts, engine)
df_counts

,table_name,row_count
0,fact_transactions,590540
1,dim_card,14893
2,dim_device,9776
3,dim_geography,438
4,dim_date,396
5,dim_email_domain,61
6,dim_product,5


## 2. Table Relationships

The fact table connects to all six dimensions through foreign keys. Note that 
`dim_email_domain` is referenced **twice** — once for the purchaser's email and once 
for the recipient's. This is a *role-playing dimension*: one canonical table serving 
two roles, avoiding duplication.

In [4]:
query_fks = """
SELECT
    kcu.column_name      AS fact_column,
    ccu.table_name       AS references_table,
    ccu.column_name      AS references_column
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu 
    ON tc.constraint_name = kcu.constraint_name
JOIN information_schema.constraint_column_usage ccu 
    ON ccu.constraint_name = tc.constraint_name
WHERE tc.constraint_type = 'FOREIGN KEY'
  AND tc.table_schema = 'fraud_dw'
  AND tc.table_name = 'fact_transactions'
ORDER BY kcu.column_name;
"""
df_fks = pd.read_sql(query_fks, engine)
df_fks

,fact_column,references_table,references_column
0,date_id,dim_date,date_id
1,device_id,dim_device,device_id
2,geography_id,dim_geography,geography_id
3,product_id,dim_product,product_id
4,purchase_email_id,dim_email_domain,email_domain_id
5,recipient_email_id,dim_email_domain,email_domain_id


## 3. Key Design Decisions

**Timestamp reconstruction.** The source stores time as an integer offset (seconds from 
a reference date), not a real timestamp. During ETL this was reconstructed into a true 
`TIMESTAMP` using `2017-12-01` as the community-validated reference date — enabling all 
temporal analysis in Phase 2.

**"Unknown" rows instead of NULL foreign keys.** ~76% of transactions have no device 
record. Rather than allowing NULL foreign keys (which break inner joins and force 
defensive NULL-handling in every query), each dimension has an explicit "unknown" row. 
Every fact row points to a valid dimension key.

**Staging layer kept separate.** Raw data lands in a `staging` schema that mirrors the 
source exactly. All cleaning happens in the transformation from staging to the `fraud_dw` 
dimensional model — keeping a clear line between "what arrived" and "what we derived."

**Type correctness.** The source's 0/1 fraud flag was converted to a proper `BOOLEAN` in 
the fact table, while staging preserved the raw integer — a deliberate staging-vs-model 
type distinction.

## 4. Data Quality & Profiling

The checks below verify the load is complete and the data is sound — the same profiling 
that precedes any trustworthy analysis.

In [5]:
query_profile = """
SELECT 
    COUNT(*)                                                    AS total_transactions,
    COUNT(*) FILTER (WHERE is_fraud)                            AS fraud_transactions,
    ROUND(100.0 * COUNT(*) FILTER (WHERE is_fraud)::numeric 
                / COUNT(*), 3)                                  AS fraud_rate_pct,
    MIN(transaction_dt)                                         AS earliest_txn,
    MAX(transaction_dt)                                         AS latest_txn,
    ROUND(MIN(transaction_amt), 2)                              AS min_amount,
    ROUND(MAX(transaction_amt), 2)                              AS max_amount,
    ROUND(AVG(transaction_amt), 2)                              AS avg_amount
FROM fraud_dw.fact_transactions;
"""
df_profile = pd.read_sql(query_profile, engine)
df_profile.T  # transpose for readability

,0
total_transactions,590540
fraud_transactions,20663
fraud_rate_pct,3.499
earliest_txn,2017-12-02 00:00:00
latest_txn,2018-06-01 23:58:51
min_amount,0.25
max_amount,31937.39
avg_amount,135.03


In [6]:
query_integrity = """
SELECT 
    COUNT(*) FILTER (WHERE date_id IS NULL)        AS null_date,
    COUNT(*) FILTER (WHERE card_id IS NULL)        AS null_card,
    COUNT(*) FILTER (WHERE product_id IS NULL)     AS null_product,
    COUNT(*) FILTER (WHERE device_id IS NULL)      AS null_device,
    COUNT(*) FILTER (WHERE geography_id IS NULL)   AS null_geography,
    COUNT(*) FILTER (WHERE purchase_email_id IS NULL)   AS null_purchaser_email,
    COUNT(*) FILTER (WHERE recipient_email_id IS NULL)  AS null_recipient_email
FROM fraud_dw.fact_transactions;
"""
df_integrity = pd.read_sql(query_integrity, engine)
print("Foreign key null counts (all should be 0):")
df_integrity.T

Foreign key null counts (all should be 0):


,0
null_date,0
null_card,0
null_product,0
null_device,0
null_geography,0
null_purchaser_email,0
null_recipient_email,0


## Summary

The warehouse holds **590,540 transactions** in a star schema with six dimensions, 
all foreign keys valid, no orphaned references. Timestamps are reconstructed, types 
are correct, and missing data is handled explicitly.

This structure supports the five analytical phases that follow:
1. **Fraud Landscape** — baseline rates and concentration
2. **Temporal Patterns** — when fraud happens
3. **Velocity Analytics** — per-card behavior
4. **Segmentation** — multi-dimensional risk profiling
5. **Rule Engine** — detection with measured performance